In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv
/kaggle/input/datasets/jakenty1/vnmathproblems2026/problem.math.csv


In [2]:
model_choice = "nemotron-3-nano:4b" # "qwen2.5:7b" #"gemma2:9b"#"mistral:7b-instruct"  # Change as needed, e.g. "gemma2:9b", "qwen2.5:7b", "llama3.1:8b"#"llama3.1:8b" #"qwen2.5:7b"  # Change to "gemma2:9b" or "llama3.1:8b" as needed
start_problem = 1
end_problem = 50
#system_prompt = None
system_prompt = "Output only the code, no explanations."

In [3]:
!head /kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv -n 1

# LLM Programming Experiment on Kaggle

This notebook allows you to run programming experiments using Ollama (Gemma 2, Llama 3.1, Qwen 2.5) on Kaggle GPU.

**Kaggle Setup:**
1. **Settings**: Enable **Internet** (required to download Ollama and models).
2. **Accelerator**: Select **GPU T4 x2** or **GPU P100**.
3. **Input**: Add the dataset `jakenty1/vnmathproblems2026` to your notebook.


In [4]:
!pip install zstandard zstd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.3 MB/s eta 0:00:00


In [5]:
!sudo apt-get install zstd




The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (11.5 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
# Install Ollama
import os
import subprocess
import time
import requests
import json
import re
import csv
from datetime import datetime

print("Installing Ollama via official script...")
# The official script is often more reliable than direct binary curl in some environments
!curl -fsSL https://ollama.com/install.sh | sh



Installing Ollama via official script...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
print("Starting Ollama server...")
# Set environment variables for Kaggle environment
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], 
                 stdout=open("ollama_stdout.log", "w"), 
                 stderr=open("ollama_stderr.log", "w"))

# Wait for server to be ready
print("Waiting for Ollama server to respond...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server ready after {i+1}s")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ERROR: Ollama server failed to start.")
    print("Last 20 lines of ollama_stderr.log:")
    !tail -n 20 ollama_stderr.log

Starting Ollama server...
Waiting for Ollama server to respond...
Ollama server ready after 2s


In [8]:
!ollama list

]11;?\NAME    ID    SIZE    MODIFIED 


## 2. Configuration and Data Loading


In [9]:
def call_ollama(model, system_prompt, user_prompt):
    start = time.time()
    try:
        payload = {"model": model, "prompt": user_prompt, "stream": False, "options": {"temperature": 0.2, "num_predict": 8192}}
        if system_prompt: payload["system"] = system_prompt
        r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        return data.get("response", ""), {"duration": time.time() - start, "prompt_tokens": data.get("prompt_eval_count", 0), "completion_tokens": data.get("eval_count", 0), "success": True, "error": ""}
    except Exception as e:
        return "", {"duration": time.time() - start, "prompt_tokens": 0, "completion_tokens": 0, "success": False, "error": str(e)}

def extract_python_code(response: str) -> str:
    matches = re.findall(r"```python\s*\n(.*?)```", response, re.DOTALL)
    if matches: return matches[-1].strip()
    matches = re.findall(r"```\s*\n(.*?)```", response, re.DOTALL)
    if matches: return matches[-1].strip()
    return response.strip()

import ast
import signal
import multiprocessing

def is_python_code(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

def _run_test_worker(code, test, result_queue):
    """Worker function to run a single test in a separate process."""
    try:
        local_ns = {}
        exec(code + "\n" + test, {}, local_ns)
        result_queue.put((True, None))
    except Exception as e:
        result_queue.put((False, str(e)))

def run_test_with_timeout(code, test, timeout=10):
    """Run a single test case with a hard timeout using multiprocessing."""
    result_queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=_run_test_worker, args=(code, test, result_queue))
    p.start()
    p.join(timeout)
    if p.is_alive():
        p.terminate()
        p.join()
        return False, f"Test timed out after {timeout}s (infinite loop or deadlock)"
    if not result_queue.empty():
        return result_queue.get()
    return False, "Test process exited unexpectedly"

def evaluate_code(code, test_list, timeout=10):
    if not code:
        return {"passed": False, "tests_passed": 0, "tests_total": len(test_list), "errors": ["No code extracted"]}
    if not is_python_code(code):
        return {"passed": False, "tests_passed": 0, "tests_total": len(test_list), "errors": ["Response is not valid Python"]}
    passed_count = 0
    errors = []
    for test in test_list:
        passed, error = run_test_with_timeout(code, test, timeout=timeout)
        if passed:
            passed_count += 1
        elif error:
            errors.append(f"Test failed: {test} | Error: {error}")
    return {"passed": passed_count == len(test_list) and len(test_list) > 0, "tests_passed": passed_count, "tests_total": len(test_list), "errors": errors}

def parse_csv_row(row):
    standard_tests = json.loads(row.get('test_cases', '[]'))
    challenge_tests = json.loads(row.get('challenge_test_list', '[]'))
    return {"task_id": f"{row['task_id']}_{row['category_id']}", "description": row['prompt'].strip(), "test_list": standard_tests + challenge_tests}

def get_completed_ids(filename):
    if not os.path.exists(filename): return set()
    completed = set()
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            try: 
                data = json.loads(line)
                if str(data.get("response_text", "")).strip():
                    completed.add(data.get("task_id"))
            except: continue
    return completed

def append_result(filename, result):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

csv_path = "/kaggle/input/datasets/jakenty1/vnmathproblems2026/problems.csv"
if not os.path.exists(csv_path): csv_path = "/kaggle/input/vnmathproblems2026/problems.csv"


In [10]:


print(f"Pulling model {model_choice}...")
!ollama pull {model_choice}

output_filename = f"/kaggle/working/{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.jsonl"

# Seed Kaggle working output from the uploaded previous-result dataset.
# Empty response rows are intentionally skipped so those tasks rerun.
seed_basename = f"{model_choice.replace(':', '-')}_{start_problem}_{end_problem}.jsonl"
seed_candidates = [
    f"/kaggle/input/nemotronresultv1/{seed_basename}",
    f"/kaggle/input/datasets/jakenty1/nemotronresultv1/{seed_basename}",
]
seed_filename = next((p for p in seed_candidates if os.path.exists(p)), None)
print("Resume seed candidates:", seed_candidates)
print("Selected seed file:", seed_filename or "NOT FOUND")

if seed_filename:
    rows_by_task_id = {}
    existing_kept_rows = 0
    existing_empty_rows = 0
    seed_added_rows = 0
    seed_duplicate_rows = 0
    seed_empty_rows = 0

    if os.path.exists(output_filename):
        with open(output_filename, "r", encoding="utf-8") as existing:
            for line in existing:
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                except Exception:
                    continue
                task_id = data.get("task_id")
                if str(data.get("response_text", "")).strip() and task_id:
                    rows_by_task_id[task_id] = data
                    existing_kept_rows += 1
                else:
                    existing_empty_rows += 1

    with open(seed_filename, "r", encoding="utf-8") as seed:
        for line in seed:
            if not line.strip():
                continue
            try:
                data = json.loads(line)
            except Exception:
                continue
            task_id = data.get("task_id")
            if not str(data.get("response_text", "")).strip():
                seed_empty_rows += 1
                continue
            if task_id in rows_by_task_id:
                seed_duplicate_rows += 1
                continue
            rows_by_task_id[task_id] = data
            seed_added_rows += 1

    with open(output_filename, "w", encoding="utf-8") as dst:
        for data in rows_by_task_id.values():
            dst.write(json.dumps(data, ensure_ascii=False) + "\n")

    print(
        f"Prepared resume file {output_filename}: "
        f"completed={len(rows_by_task_id)}, existing_kept={existing_kept_rows}, "
        f"existing_empty_removed={existing_empty_rows}, seed_added={seed_added_rows}, "
        f"seed_duplicate={seed_duplicate_rows}, seed_empty_skipped={seed_empty_rows}."
    )
else:
    try:
        print("Available /kaggle/input directories:", os.listdir("/kaggle/input"))
    except Exception as e:
        print("Could not list /kaggle/input:", e)

completed_ids = get_completed_ids(output_filename)

problems_to_test = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            if start_problem <= int(row['task_id']) <= end_problem:
                prob = parse_csv_row(row)
                if prob['task_id'] not in completed_ids:
                    problems_to_test.append(prob)
        except: continue

print(f"Loaded {len(problems_to_test)} remaining tasks for range {start_problem}-{end_problem}.")

for prob in problems_to_test:
    print(f"Testing {prob['task_id']}...", end=" ", flush=True)
    response_text, meta = call_ollama(model_choice, system_prompt, prob['description'])
    if not meta['success']:
        print(f"Error: {meta['error']}")
        continue
    
    extracted = extract_python_code(response_text)
    eval_res = evaluate_code(extracted, prob['test_list'])
    print(f"{'PASS' if eval_res['passed'] else 'FAIL'} ({eval_res['tests_passed']}/{eval_res['tests_total']})")
    
    result = {
        "timestamp": datetime.now().isoformat(), "model": model_choice, "task_id": prob['task_id'],
        "passed": eval_res['passed'], "tests_passed": eval_res['tests_passed'], "tests_total": eval_res['tests_total'],
        "errors": eval_res['errors'], "duration_seconds": meta['duration'],
        "prompt_tokens": meta['prompt_tokens'], "completion_tokens": meta['completion_tokens'],
        "response_text": response_text, "extracted_code": extracted
    }
    append_result(output_filename, result)

print(f"Results saved to {output_filename}")


Pulling model nemotron-3-nano:4b...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 527db2cf6c70:   0% ▕                  ▏ 9.2 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   3% ▕                  ▏  91 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   7% ▕█                 ▏ 187 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:   8% ▕█                 ▏ 226 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  11% ▕██                ▏ 317 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  15% ▕██                ▏ 414 MB/2.8 GB                  pulling manifest 
pulling 527db2cf6c70:  16% ▕██                ▏ 463 MB/2.8 GB          

## 4. Run Additional Ranges
Edit the problem range below and rerun this cell to process new tasks without restarting.